# Results for model: qwen/qwen2-7b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

def train_model():
    # Load data
    data_path = './jane_street_data/train.parquet'
    data = pl.read_parquet(data_path)

    # Feature Engineering
    TOP_QUANTILE = 0.15
    data['date_order_rank'] = data['date_id'].rank()
    data = data.sort('date_id')
    data['rolling_top_quantile_rank'] = data['feature_00'].rank() < (TOP_QUANTILE * 100)
    data = data.drop(['date_order_rank'])
    data_count = data.shape[0]

    # Prepare rolling batches
    train_size = int(data_count * 0.8)
    val_size = int(data_count * 0.1)
    test_size = data_count - train_size - val_size
    data = data.with_columns([
        pl.lit(1).alias('is_train') if row['date_id'] <= (train_size + val_size) else pl.lit(0).alias('is_train')
        if row['date_id'] <= (train_size + val_size) else pl.lit(1).alias('is_train')
        for row in data.iter_rows()
    ]).to_pandas()
    rolling_bootstrap_data = xgb.util.rolling_bootstrap(data)

    # Prepare datasets
    X = rolling_bootstrap_data['rolling_top_quantile_rank']
    y = rolling_bootstrap_data['responder_6']
    x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.1, shuffle=False)

    # Model training
    params = {
        'max_depth': 5,
        'occupation': 1,
        'gamma': 2,
        'primary_group': 3,
        'eta': 0.025,
        'reg_lambda': 1,
        'objective': 'reg:squaredlogerror'
    }
    xg_clf = XGBRegressor(**params)
    xg_clf.fit(x_train, y_train)

    # Model evaluation
    predicted_y_val = xg_clf.predict(x_val)
    mse_val = mean_squared_log_error(y_val, predicted_y_val)
    print('Evaluated Model MSE:', mse_val)

train_model()